# Binary PneumoniaMNIST Training
3-epoch CPU baseline — binary classification (Normal vs Pneumonia).  
No checkpoint saving. No external logging. No GPU.

In [ ]:
# ─── Section 1: Imports and Config ────────────────────────────────────────────
import os
import sys

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from qcore.data.registry import get_dataset
from qcore.data.torch_adapter import TorchDatasetAdapter

EPOCHS     = 3
BATCH_SIZE = 32
LR         = 1e-3
DEVICE     = 'cpu'

print(f'EPOCHS     : {EPOCHS}')
print(f'BATCH_SIZE : {BATCH_SIZE}')
print(f'LR         : {LR}')
print(f'DEVICE     : {DEVICE}')
print(f'torch      : {torch.__version__}')

In [ ]:
# ─── Section 2: Load Datasets and DataLoaders ─────────────────────────────────
print('Loading datasets ...')

train_ds = get_dataset('pneumoniamnist', 'train')
val_ds   = get_dataset('pneumoniamnist', 'val')
test_ds  = get_dataset('pneumoniamnist', 'test')

train_adapted = TorchDatasetAdapter(train_ds)
val_adapted   = TorchDatasetAdapter(val_ds)
test_adapted  = TorchDatasetAdapter(test_ds)

train_loader = DataLoader(train_adapted, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_adapted,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_adapted,  batch_size=BATCH_SIZE, shuffle=False)

print(f'Train samples : {len(train_adapted)}')
print(f'Val   samples : {len(val_adapted)}')
print(f'Test  samples : {len(test_adapted)}')
print(f'Train batches : {len(train_loader)}')
print(f'Val   batches : {len(val_loader)}')
print(f'Test  batches : {len(test_loader)}')

In [ ]:
# ─── Section 3: Visualize Sample Batch ────────────────────────────────────────
label_map = train_ds.label_map

x_batch, y_batch = next(iter(train_loader))
print(f'Batch shape : {x_batch.shape}  dtype={x_batch.dtype}')
print(f'Label shape : {y_batch.shape}  dtype={y_batch.dtype}')

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for ax, i in zip(axes.flat, range(8)):
    img_np = x_batch[i].cpu().numpy().squeeze()   # (1, H, W) → (H, W)
    lbl    = int(y_batch[i].item())
    ax.imshow(img_np, cmap='gray', vmin=0, vmax=1)
    ax.set_title(label_map.get(lbl, str(lbl)), fontsize=8)
    ax.axis('off')

plt.suptitle('PneumoniaMNIST — sample train batch', fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'reports', 'pneumoniamnist_sample_batch.png'), dpi=120)
plt.show()
print('Sample batch plot saved.')

In [ ]:
# ─── Section 4: Define Tiny CNN (inline) ──────────────────────────────────────
model = nn.Sequential(
    nn.Conv2d(1, 8, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.AdaptiveAvgPool2d((1, 1)),
    nn.Flatten(),
    nn.Linear(8, 2),
)
model.to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f'Total parameters : {n_params}')

In [ ]:
# ─── Section 5: Train Loop ─────────────────────────────────────────────────────
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn   = nn.CrossEntropyLoss()

train_losses = []
val_losses   = []
train_accs   = []
val_accs     = []

for epoch in range(1, EPOCHS + 1):
    # ── Train pass ────────────────────────────────────────────────────────────
    model.train()
    running_loss = 0.0
    n_correct    = 0
    n_total      = 0

    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb)
        loss   = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * xb.size(0)
        preds         = logits.argmax(dim=1)
        n_correct    += (preds == yb).sum().item()
        n_total      += xb.size(0)

    epoch_train_loss = running_loss / n_total
    epoch_train_acc  = n_correct / n_total

    # ── Val pass ──────────────────────────────────────────────────────────────
    model.eval()
    val_running_loss = 0.0
    val_n_correct    = 0
    val_n_total      = 0

    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits  = model(xb)
            loss    = loss_fn(logits, yb)

            val_running_loss += loss.item() * xb.size(0)
            preds             = logits.argmax(dim=1)
            val_n_correct    += (preds == yb).sum().item()
            val_n_total      += xb.size(0)

    epoch_val_loss = val_running_loss / val_n_total
    epoch_val_acc  = val_n_correct / val_n_total

    train_losses.append(epoch_train_loss)
    val_losses.append(epoch_val_loss)
    train_accs.append(epoch_train_acc)
    val_accs.append(epoch_val_acc)

    print(
        f'Epoch {epoch}/{EPOCHS} | '
        f'train loss={epoch_train_loss:.4f}  acc={epoch_train_acc:.4f} | '
        f'val loss={epoch_val_loss:.4f}  acc={epoch_val_acc:.4f}'
    )

print('Training complete.')

In [ ]:
# ─── Section 6: Plot Loss Curve ───────────────────────────────────────────────
epochs_axis = list(range(1, EPOCHS + 1))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(epochs_axis, train_losses, marker='o', label='Train loss')
ax.plot(epochs_axis, val_losses,   marker='s', label='Val loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Cross-entropy loss')
ax.set_title('PneumoniaMNIST — loss curves')
ax.legend()
ax.set_xticks(epochs_axis)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'reports', 'pneumoniamnist_loss_curve.png'), dpi=120)
plt.show()
print('Loss curve saved.')

In [ ]:
# ─── Section 7: Plot Accuracy Curve ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(epochs_axis, [a * 100 for a in train_accs], marker='o', label='Train accuracy')
ax.plot(epochs_axis, [a * 100 for a in val_accs],   marker='s', label='Val accuracy')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy (%)')
ax.set_title('PneumoniaMNIST — accuracy curves')
ax.legend()
ax.set_xticks(epochs_axis)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'reports', 'pneumoniamnist_acc_curve.png'), dpi=120)
plt.show()
print('Accuracy curve saved.')

In [ ]:
# ─── Section 8: Evaluate on Test Split ────────────────────────────────────────
model.eval()
test_n_correct = 0
test_n_total   = 0

with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb  = xb.to(DEVICE), yb.to(DEVICE)
        logits   = model(xb)
        preds    = logits.argmax(dim=1)
        test_n_correct += (preds == yb).sum().item()
        test_n_total   += xb.size(0)

test_accuracy = test_n_correct / test_n_total
print(f'Test accuracy : {test_accuracy * 100:.2f}%  ({test_n_correct}/{test_n_total})')

In [ ]:
# ─── Section 9: Confusion Matrix ──────────────────────────────────────────────
all_preds  = []
all_labels = []

model.eval()
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(DEVICE)
        logits = model(xb)
        preds  = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(yb.numpy().tolist())

class_names = [label_map.get(i, str(i)) for i in sorted(label_map.keys())]
cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('PneumoniaMNIST — test confusion matrix')
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'reports', 'pneumoniamnist_confusion_matrix.png'), dpi=120)
plt.show()
print('Confusion matrix saved.')

In [ ]:
# ─── Section 10: Sample Predictions ───────────────────────────────────────────
model.eval()
x_test_batch, y_test_batch = next(iter(test_loader))

with torch.no_grad():
    test_logits = model(x_test_batch.to(DEVICE))
    test_preds  = test_logits.argmax(dim=1)

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for ax, i in zip(axes.flat, range(8)):
    img_np = x_test_batch[i].cpu().numpy().squeeze()   # (1, H, W) → (H, W)
    pred   = int(test_preds[i].item())
    true   = int(y_test_batch[i].item())
    pred_name = label_map.get(pred, str(pred))
    true_name = label_map.get(true, str(true))
    color = 'green' if pred == true else 'red'
    ax.imshow(img_np, cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'P:{pred_name}\nT:{true_name}', fontsize=7, color=color)
    ax.axis('off')

plt.suptitle('PneumoniaMNIST — sample test predictions (green=correct, red=wrong)', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'reports', 'pneumoniamnist_sample_preds.png'), dpi=120)
plt.show()
print('Sample predictions plot saved.')

In [ ]:
# ─── Section 11: GO / NO-GO Summary ───────────────────────────────────────────
GO_THRESHOLD = 0.70

print('=== GO / NO-GO Summary ===')
print()
print(f'  Test accuracy : {test_accuracy * 100:.2f}%')
print()

if test_accuracy >= GO_THRESHOLD:
    verdict = 'GO'
else:
    verdict = 'NO-GO'

print(f'  Verdict : {verdict}  (threshold >= {GO_THRESHOLD * 100:.0f}%)')
print()
print('  Note: 3-epoch CPU baseline only — not a production model')
print()
print(f'>>> {verdict} — binary PneumoniaMNIST baseline complete.')